## Load conditionwise_stats and stimulus_conditions

In [1]:
import numpy as np
import pandas as pd
import scipy.stats
import warnings
from pathlib import Path
from scipy.optimize import curve_fit

In [2]:
results_dir = Path(r"C:\Users\MaryBeth\projects\SarvestaniLab\OpenScopeMouseV1\jupyternotebooks\ephys\results")

conditionwise_stats = pd.read_csv(results_dir / "ephys_conditionwise_stats.csv")
stimulus_conditions = pd.read_csv(results_dir / "ephys_stimulus_conditions.csv")

# Re-index stimulus_conditions on stimulus_condition_id
stimulus_conditions = stimulus_conditions.set_index('stimulus_condition_id')

# Set multi-index on conditionwise_stats for fast lookup
conditionwise_stats = conditionwise_stats.set_index(['unit_id', 'stimulus_condition_id'])

print(f"conditionwise_stats: {conditionwise_stats.shape}")
print(f"stimulus_conditions: {stimulus_conditions.shape}")
print(conditionwise_stats.head())
print(stimulus_conditions.head())

conditionwise_stats: (2037400, 8)
stimulus_conditions: (100, 4)
                                                 spike_count  \
unit_id                   stimulus_condition_id                
sub-810531__ProbeA__unit0 0                               10   
sub-810531__ProbeA__unit1 0                               11   
sub-810531__ProbeA__unit2 0                               29   
sub-810531__ProbeA__unit3 0                               31   
sub-810531__ProbeA__unit4 0                                7   

                                                 stimulus_presentation_count  \
unit_id                   stimulus_condition_id                                
sub-810531__ProbeA__unit0 0                                               15   
sub-810531__ProbeA__unit1 0                                               15   
sub-810531__ProbeA__unit2 0                                               15   
sub-810531__ProbeA__unit3 0                                               15   
sub-810

## Define metric functions

In [3]:
def deg2rad(arr):
    
    return arr / 180 * np.pi

# from lines 812 - 840 of AllenSDK/allensdk/brain_observatory/ecephys/stimulus_analysis/stimulus_analysis.py
# https://github.com/AllenInstitute/AllenSDK/blob/a9b5c685396126d9748f1ccecf7c00f440569f69/allensdk/brain_observatory/ecephys/stimulus_analysis/stimulus_analysis.py
def osi(orivals, tuning):
    if len(orivals) == 0 or len(orivals) != len(tuning):
        warnings.warn('orivals and tunings are of different lengths')
        return np.nan
    tuning_sum = tuning.sum()
    if tuning_sum == 0.0:
        return np.nan
    cv_top = tuning * np.exp(1j * 2 * orivals)
    
    return np.abs(cv_top.sum()) / tuning_sum

# from lines 843 - 860 of AllenSDK/allensdk/brain_observatory/ecephys/stimulus_analysis/stimulus_analysis.py
# https://github.com/AllenInstitute/AllenSDK/blob/a9b5c685396126d9748f1ccecf7c00f440569f69/allensdk/brain_observatory/ecephys/stimulus_analysis/stimulus_analysis.py
def dsi(orivals, tuning):
    if len(orivals) == 0 or len(orivals) != len(tuning):
        warnings.warn('orivals and tunings are of different lengths')
        return np.nan
    tuning_sum = tuning.sum()
    if tuning_sum == 0.0:
        return np.nan
    cv_top = tuning * np.exp(1j * orivals)
    
    return np.abs(cv_top.sum()) / tuning_sum

# from lines 290 - 319 of AllenSDK/allensdk/brain_observatory/ecephys/stimulus_analysis/drifting_gratings.py
# https://github.com/AllenInstitute/AllenSDK/blob/a9b5c685396126d9748f1ccecf7c00f440569f69/allensdk/brain_observatory/ecephys/stimulus_analysis/drifting_gratings.py
def calculate_osi_dsi(unit_id, pref_tf, conditionwise_stats, stimulus_conditions):
    condition_inds = stimulus_conditions[stimulus_conditions['temporal_frequency'] == pref_tf].index.values
    df = conditionwise_stats.loc[unit_id].loc[condition_inds].copy()
    df = df.assign(ori=stimulus_conditions.loc[df.index.values, 'orientation'])
    tuning = df.groupby('ori')['spike_mean'].mean().sort_index().values
    unique_oris = np.sort(df['ori'].unique())
    orivals_rad = deg2rad(unique_oris).astype('complex128')
    
    return osi(orivals_rad, tuning), dsi(orivals_rad, tuning)


def calculate_osi_dsi_all_tf(unit_id, conditionwise_stats, stimulus_conditions):
    """
    Calculate OSI and DSI averaged across all temporal frequencies.
    Used when preferred TF is a continuous float (e.g. from Gaussian fit)
    and cannot be matched to a discrete condition.
    """
    all_condition_inds = stimulus_conditions.index.values
    df = conditionwise_stats.loc[unit_id].loc[all_condition_inds].copy()
    df = df.assign(ori=stimulus_conditions.loc[df.index.values, 'orientation'])
    tuning = df.groupby('ori')['spike_mean'].mean().sort_index().values
    unique_oris = np.sort(df['ori'].unique())
    orivals_rad = deg2rad(unique_oris).astype('complex128')
    
    return osi(orivals_rad, tuning), dsi(orivals_rad, tuning)


def gaussian_2d_sftf(coords, amplitude, log_sf0, log_tf0, sigma_sf, sigma_tf, offset):
    log_sf, log_tf = coords
    g = offset + amplitude * np.exp(
        -(
            (log_sf - log_sf0) ** 2 / (2 * sigma_sf ** 2) +
            (log_tf - log_tf0) ** 2 / (2 * sigma_tf ** 2)
        )
    )
    
    return g.ravel()


def fit_sftf_gaussian(sf_vals, tf_vals, response_matrix):
    sf_vals = np.array(sf_vals, dtype=float)
    tf_vals = np.array(tf_vals, dtype=float)
    response_matrix = np.array(response_matrix, dtype=float)

    log_sf = np.log2(sf_vals)
    log_tf = np.log2(tf_vals)
    log_sf_grid, log_tf_grid = np.meshgrid(log_sf, log_tf, indexing='ij')
    z = response_matrix.ravel()

    flat_idx = np.argmax(response_matrix)
    sf_idx, tf_idx = np.unravel_index(flat_idx, response_matrix.shape)

    amplitude_guess = response_matrix.max() - response_matrix.min()
    initial_guess = (
        amplitude_guess, log_sf[sf_idx], log_tf[tf_idx],
        (log_sf[-1] - log_sf[0]) / 4.0 if len(log_sf) > 1 else 1.0,
        (log_tf[-1] - log_tf[0]) / 4.0 if len(log_tf) > 1 else 1.0,
        response_matrix.min()
    )
    bounds = (
        [0,      log_sf[0],  log_tf[0],  0.1, 0.1, -np.inf],
        [np.inf, log_sf[-1], log_tf[-1], 10,  10,   np.inf]
    )
    coords = (log_sf_grid.ravel(), log_tf_grid.ravel())

    def _r2(z_data, z_fit):
        ss_res = np.sum((z_data - z_fit) ** 2)
        ss_tot = np.sum((z_data - np.mean(z_data)) ** 2)
        return 1 - (ss_res / ss_tot) if ss_tot != 0 else 0.0

    try:
        popt, _ = curve_fit(gaussian_2d_sftf, coords, z, p0=initial_guess, bounds=bounds, maxfev=10000)
        r2 = _r2(z, gaussian_2d_sftf(coords, *popt))
        if r2 < -0.5:
            return None, None, None, None
        return 2 ** popt[1], 2 ** popt[2], r2, popt
    except RuntimeError:
        try:
            popt, _ = curve_fit(gaussian_2d_sftf, coords, z, p0=initial_guess, maxfev=10000)
            r2 = _r2(z, gaussian_2d_sftf(coords, *popt))
            return 2 ** popt[1], 2 ** popt[2], r2, popt
        except Exception:
            return None, None, None, None
    except Exception:
        return None, None, None, None


def bin_to_nearest(value, presented_values):
    if value is None or np.isnan(value):
        return np.nan
    presented_values = np.array(presented_values)
    return presented_values[np.argmin(np.abs(np.log2(presented_values) - np.log2(value)))]

## Compute preferred variables for all units

In [4]:
ori_vals = np.sort(stimulus_conditions['orientation'].unique())
tf_vals  = np.sort(stimulus_conditions['temporal_frequency'].unique())
sf_vals  = np.sort(stimulus_conditions['spatial_frequency'].unique())

unit_ids = conditionwise_stats.index.get_level_values('unit_id').unique()
print(f"Units to process: {len(unit_ids)}")
print(f"Orientations: {ori_vals}")
print(f"Temporal frequencies: {tf_vals}")
print(f"Spatial frequencies: {sf_vals}")

Units to process: 20374
Orientations: [  0.  45.  90. 135.]
Temporal frequencies: [ 1.  2.  4.  8. 15.]
Spatial frequencies: [0.02 0.04 0.08 0.16 0.32]


In [5]:
results = []

for i, unit_id in enumerate(unit_ids):
    if i % 500 == 0:
        print(f"  {i}/{len(unit_ids)}...")

    try:
        unit_stats = conditionwise_stats.loc[unit_id]

        def mean_response(conditions):
            try:
                return float(unit_stats.loc[conditions, 'spike_mean'].mean())
            except (KeyError, TypeError):
                return 0.0

        # Step 1: pref_ori — argmax across all TFs and SFs
        ori_responses = [
            mean_response(stimulus_conditions[stimulus_conditions['orientation'] == ori].index.values)
            for ori in ori_vals
        ]
        pref_ori = ori_vals[np.argmax(ori_responses)]

        # Step 2: pref_sf — argmax across all conditions
        sf_responses = [
            mean_response(stimulus_conditions[stimulus_conditions['spatial_frequency'] == sf].index.values)
            for sf in sf_vals
        ]
        pref_sf = sf_vals[np.argmax(sf_responses)]

        # Step 3: pref_tf — argmax across all conditions
        tf_responses = [
            mean_response(stimulus_conditions[stimulus_conditions['temporal_frequency'] == tf].index.values)
            for tf in tf_vals
        ]
        pref_tf = tf_vals[np.argmax(tf_responses)]

        # Step 4: pref_tf_nested — at pref_ori & pref_sf
        tf_responses_nested = [
            mean_response(stimulus_conditions[
                (stimulus_conditions['orientation'] == pref_ori) &
                (stimulus_conditions['spatial_frequency'] == pref_sf) &
                (stimulus_conditions['temporal_frequency'] == tf)
            ].index.values)
            for tf in tf_vals
        ]
        pref_tf_nested = tf_vals[np.argmax(tf_responses_nested)]

        # Step 5: pref_sf_nested — at pref_ori & pref_tf
        sf_responses_nested = [
            mean_response(stimulus_conditions[
                (stimulus_conditions['orientation'] == pref_ori) &
                (stimulus_conditions['temporal_frequency'] == pref_tf) &
                (stimulus_conditions['spatial_frequency'] == sf)
            ].index.values)
            for sf in sf_vals
        ]
        pref_sf_nested = sf_vals[np.argmax(sf_responses_nested)]

        # Step 6: Gaussian fit in SF x TF space at pref_ori
        sftf_matrix = np.array([
            [
                mean_response(stimulus_conditions[
                    (stimulus_conditions['orientation'] == pref_ori) &
                    (stimulus_conditions['spatial_frequency'] == sf) &
                    (stimulus_conditions['temporal_frequency'] == tf)
                ].index.values)
                for tf in tf_vals
            ]
            for sf in sf_vals
        ])
        pref_sf_gaussian, pref_tf_gaussian, r2_gaussian, _ = fit_sftf_gaussian(sf_vals, tf_vals, sftf_matrix)
        pref_sf_gaussian_snapped = bin_to_nearest(pref_sf_gaussian, sf_vals)
        pref_tf_gaussian_snapped = bin_to_nearest(pref_tf_gaussian, tf_vals)

        # Step 7: OSI / DSI
        osi_val,        dsi_val        = calculate_osi_dsi(unit_id, pref_tf,        conditionwise_stats, stimulus_conditions)
        osi_val_nested, dsi_val_nested = calculate_osi_dsi(unit_id, pref_tf_nested, conditionwise_stats, stimulus_conditions)
        osi_val_gaussian, dsi_val_gaussian = (
            calculate_osi_dsi_all_tf(unit_id, conditionwise_stats, stimulus_conditions)
            if pref_tf_gaussian is not None else (np.nan, np.nan)
        )

        # Pull mouse_name / probe from conditionwise_stats columns
        row_meta = conditionwise_stats.loc[unit_id].iloc[0]

        results.append({
            'unit_id':                  unit_id,
            'mouse_name':               row_meta.get('mouse_name', np.nan),
            'probe':                    row_meta.get('probe', np.nan),
            'raw_unit_id':              row_meta.get('raw_unit_id', np.nan),
            'pref_ori':                 float(pref_ori),
            'pref_sf':                  float(pref_sf),
            'pref_tf':                  float(pref_tf),
            'pref_tf_nested':           float(pref_tf_nested),
            'pref_sf_nested':           float(pref_sf_nested),
            'pref_sf_gaussian':         float(pref_sf_gaussian) if pref_sf_gaussian is not None else np.nan,
            'pref_tf_gaussian':         float(pref_tf_gaussian) if pref_tf_gaussian is not None else np.nan,
            'pref_sf_gaussian_snapped': pref_sf_gaussian_snapped,
            'pref_tf_gaussian_snapped': pref_tf_gaussian_snapped,
            'r_squared_gaussian':       float(r2_gaussian) if r2_gaussian is not None else np.nan,
            'osi_dg':                   float(osi_val),
            'dsi_dg':                   float(dsi_val),
            'osi_dg_nested':            float(osi_val_nested),
            'dsi_dg_nested':            float(dsi_val_nested),
            'osi_dg_gaussian':          float(osi_val_gaussian) if not np.isnan(osi_val_gaussian) else np.nan,
            'dsi_dg_gaussian':          float(dsi_val_gaussian) if not np.isnan(dsi_val_gaussian) else np.nan,
            'peak_spike_mean':          float(max(ori_responses)),
        })

    except Exception as e:
        import traceback
        print(f"  ERROR on {unit_id}: {e}")
        traceback.print_exc()
        continue

results_df = pd.DataFrame(results)
print(f"\nDone. {len(results_df)} units processed.")
print(results_df.head())

  0/20374...
  500/20374...


c:\Users\MaryBeth\anaconda3\envs\openscope\lib\site-packages\scipy\optimize\_minpack_py.py:906: OptimizeWarning: Covariance of the parameters could not be estimated
  warnings.warn('Covariance of the parameters could not be estimated',


  1000/20374...
  1500/20374...
  2000/20374...
  2500/20374...
  3000/20374...
  3500/20374...
  4000/20374...
  4500/20374...
  5000/20374...
  5500/20374...
  6000/20374...
  6500/20374...
  7000/20374...
  7500/20374...
  8000/20374...
  8500/20374...
  9000/20374...
  9500/20374...
  10000/20374...
  10500/20374...
  11000/20374...
  11500/20374...
  12000/20374...
  12500/20374...
  13000/20374...
  13500/20374...
  14000/20374...
  14500/20374...
  15000/20374...
  15500/20374...
  16000/20374...


c:\Users\MaryBeth\anaconda3\envs\openscope\lib\site-packages\scipy\optimize\_minpack_py.py:906: OptimizeWarning: Covariance of the parameters could not be estimated
  warnings.warn('Covariance of the parameters could not be estimated',


  16500/20374...
  17000/20374...
  17500/20374...
  18000/20374...
  18500/20374...
  19000/20374...
  19500/20374...
  20000/20374...

Done. 20374 units processed.
                     unit_id  mouse_name   probe  raw_unit_id  pref_ori  \
0  sub-810531__ProbeA__unit0  sub-810531  ProbeA            0      45.0   
1  sub-810531__ProbeA__unit1  sub-810531  ProbeA            1      45.0   
2  sub-810531__ProbeA__unit2  sub-810531  ProbeA            2      90.0   
3  sub-810531__ProbeA__unit3  sub-810531  ProbeA            3      45.0   
4  sub-810531__ProbeA__unit4  sub-810531  ProbeA            4      90.0   

   pref_sf  pref_tf  pref_tf_nested  pref_sf_nested  pref_sf_gaussian  ...  \
0     0.32      4.0             4.0            0.32          0.301700  ...   
1     0.32      4.0             8.0            0.32          0.044821  ...   
2     0.32      4.0             8.0            0.16          0.320000  ...   
3     0.32      4.0             4.0            0.32          0.320000  

## Save results

In [6]:
results_df.to_csv(results_dir / "ephys_pref_variables.csv", index=False)
print(f"Saved: ephys_pref_variables.csv  ({len(results_df)} units x {len(results_df.columns)} columns)")
print(results_df.columns.tolist())

Saved: ephys_pref_variables.csv  (20374 units x 21 columns)
['unit_id', 'mouse_name', 'probe', 'raw_unit_id', 'pref_ori', 'pref_sf', 'pref_tf', 'pref_tf_nested', 'pref_sf_nested', 'pref_sf_gaussian', 'pref_tf_gaussian', 'pref_sf_gaussian_snapped', 'pref_tf_gaussian_snapped', 'r_squared_gaussian', 'osi_dg', 'dsi_dg', 'osi_dg_nested', 'dsi_dg_nested', 'osi_dg_gaussian', 'dsi_dg_gaussian', 'peak_spike_mean']
